In [13]:
import datetime
import polars as pl
import numpy as np
from utils import find_contract_expiry

In [14]:
etf_returns = pl.read_excel("./etf returns.xlsx")
# display(etf_returns.head(2))

cboe_vx = pl.read_excel("./cboe vx.xlsx")
# display(cboe_vx.head(2))

In [15]:
left = etf_returns.select([pl.col("column_0").alias("date"), "BATS:VIXY - Return"])
right = etf_returns.select(["_1", "ARCX:SPY - Return"])
etf_returns = left.join(right, left_on="date", right_on="_1", how="inner")
etf_returns = etf_returns.sort(by="date")
etf_returns = etf_returns.rename({"date": "Trade Date"})

print(f"vx zero close prices: {cboe_vx.select(pl.col('Close') == 0).sum().item()}")
# cloes prices are actually handled differently in futures markets. if there are no trades in the last minutes of the trading day, the exchange will mark the close as 0.
# lets just use the settle price, which is apparently the preferred price metric used in futures markets
cboe_vx = cboe_vx.select([
    pl.col("Trade Date"), 
    pl.col("Futures").alias("Contract Date"), 
    pl.col("Futures")
        .str.extract(r"\((.*)\)", 1) # fuck regex
        .str.to_date(format="%b %Y")
        .map_elements(find_contract_expiry, return_dtype=pl.Date)
        .alias("Contract Expiry"), 
    pl.col("Settle")
])
cboe_vx = cboe_vx.with_columns((pl.col("Contract Expiry") - pl.col("Trade Date")).dt.total_days().alias("Days to Expiry"))

cboe_vx = cboe_vx.sort(by="Trade Date")

display(etf_returns.head(2))
display(cboe_vx.head(2))

vx zero close prices: 75


Trade Date,BATS:VIXY - Return,ARCX:SPY - Return
date,f64,f64
2023-03-17,10.5315,-1.16478
2023-03-20,-4.18522,0.96156


Trade Date,Contract Date,Contract Expiry,Settle,Days to Expiry
date,str,date,f64,i64
2024-04-22,"""F (Jan 2025)""",2025-01-22,19.25,275
2024-04-23,"""F (Jan 2025)""",2025-01-22,18.7,274


In [16]:
dates = cboe_vx.select(pl.col("Trade Date")).unique()

from utils import interp_targets
curve_table = cboe_vx.group_by("Trade Date").map_groups(interp_targets).sort("Trade Date").drop_nulls()
curve_table = curve_table.with_columns([(pl.col("t_7") / pl.col("t_30")).alias("7/30 ratio"), (pl.col("t_30") / pl.col("t_60")).alias("30/60 ratio"), (pl.col("t_30") / pl.col("t_90")).alias("30/90 ratio"), (pl.col("t_30") / pl.col("t_120")).alias("30/120 ratio")])
curve_table = curve_table.with_columns([(pl.col("t_7") > pl.col("t_30")).alias("7/30 flag"), (pl.col("t_30") > pl.col("t_60")).alias("30/60 flag"), (pl.col("t_30") > pl.col("t_90")).alias("30/90 flag"), (pl.col("t_30") > pl.col("t_120")).alias("30/120 flag")])
curve_table

Trade Date,t_7,t_30,t_60,t_90,t_120,7/30 ratio,30/60 ratio,30/90 ratio,30/120 ratio,7/30 flag,30/60 flag,30/90 flag,30/120 flag
date,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool
2024-05-28,14.725,15.0125,15.3875,15.7625,16.1375,0.980849,0.97563,0.952419,0.930287,false,false,false,false
2024-05-29,14.26875,14.617857,15.073214,15.528571,15.983929,0.976118,0.96979,0.941352,0.914535,false,false,false,false
2024-05-30,14.594643,14.902679,15.304464,15.70625,16.108036,0.97933,0.973747,0.948837,0.92517,false,false,false,false
2024-05-31,13.844643,14.214286,14.696429,15.178571,15.660714,0.973995,0.967193,0.936471,0.90764,false,false,false,false
2024-06-03,14.296429,14.625,15.053571,15.482143,15.910714,0.977534,0.97153,0.944637,0.919192,false,false,false,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-03-09,23.608457,23.5673,22.517793,21.900436,21.283079,1.001746,1.046608,1.076111,1.107326,true,true,true,true
2026-03-10,25.137354,25.0953,22.908514,21.646907,20.3853,1.001676,1.095457,1.159302,1.231049,true,true,true,true
2026-03-11,23.5823,23.5823,22.381282,21.701461,21.021639,1.0,1.053662,1.086669,1.121811,false,true,true,true


In [17]:
main = etf_returns.join(curve_table, on="Trade Date")
main

Trade Date,BATS:VIXY - Return,ARCX:SPY - Return,t_7,t_30,t_60,t_90,t_120,7/30 ratio,30/60 ratio,30/90 ratio,30/120 ratio,7/30 flag,30/60 flag,30/90 flag,30/120 flag
date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool
2024-05-28,2.51346,0.06989,14.725,15.0125,15.3875,15.7625,16.1375,0.980849,0.97563,0.952419,0.930287,false,false,false,false
2024-05-29,3.67776,-0.70025,14.26875,14.617857,15.073214,15.528571,15.983929,0.976118,0.96979,0.941352,0.914535,false,false,false,false
2024-05-30,-0.16892,-0.66337,14.594643,14.902679,15.304464,15.70625,16.108036,0.97933,0.973747,0.948837,0.92517,false,false,false,false
2024-05-31,-3.21489,0.91081,13.844643,14.214286,14.696429,15.178571,15.660714,0.973995,0.967193,0.936471,0.90764,false,false,false,false
2024-06-03,-0.52448,0.08154,14.296429,14.625,15.053571,15.482143,15.910714,0.977534,0.97153,0.944637,0.919192,false,false,false,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-03-09,-9.52104,0.87599,23.608457,23.5673,22.517793,21.900436,21.283079,1.001746,1.046608,1.076111,1.107326,true,true,true,true
2026-03-10,4.8444,-0.1607,25.137354,25.0953,22.908514,21.646907,20.3853,1.001676,1.095457,1.159302,1.231049,true,true,true,true
2026-03-11,-4.55936,-0.12552,23.5823,23.5823,22.381282,21.701461,21.021639,1.0,1.053662,1.086669,1.121811,false,true,true,true


In [18]:
from utils import add_cumret
short = main
main = add_cumret(main, "7/30 ratio", "BATS:VIXY - Return")
main = add_cumret(main, "30/60 ratio", "BATS:VIXY - Return")
main = add_cumret(main, "30/90 ratio", "BATS:VIXY - Return")
main = add_cumret(main, "30/120 ratio", "BATS:VIXY - Return")
main

Trade Date,BATS:VIXY - Return,ARCX:SPY - Return,t_7,t_30,t_60,t_90,t_120,7/30 ratio,30/60 ratio,30/90 ratio,30/120 ratio,7/30 flag,30/60 flag,30/90 flag,30/120 flag,return earned: 7/30 ratio (shifted),cumret: 7/30 ratio,return earned: 30/60 ratio (shifted),cumret: 30/60 ratio,return earned: 30/90 ratio (shifted),cumret: 30/90 ratio,return earned: 30/120 ratio (shifted),cumret: 30/120 ratio
date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64
2024-05-28,2.51346,0.06989,14.725,15.0125,15.3875,15.7625,16.1375,0.980849,0.97563,0.952419,0.930287,false,false,false,false,null,null,null,null,null,null,null,null
2024-05-29,3.67776,-0.70025,14.26875,14.617857,15.073214,15.528571,15.983929,0.976118,0.96979,0.941352,0.914535,false,false,false,false,3.607328,1.036073,3.588131,1.035881,3.502767,1.035028,3.421371,1.034214
2024-05-30,-0.16892,-0.66337,14.594643,14.902679,15.304464,15.70625,16.108036,0.97933,0.973747,0.948837,0.92517,false,false,false,false,-0.164886,1.034365,-0.163817,1.034184,-0.159013,1.033382,-0.154483,1.032616
2024-05-31,-3.21489,0.91081,13.844643,14.214286,14.696429,15.178571,15.660714,0.973995,0.967193,0.936471,0.90764,false,false,false,false,-3.148439,1.001799,-3.13049,1.001809,-3.050408,1.001859,-2.974321,1.001903
2024-06-03,-0.52448,0.08154,14.296429,14.625,15.053571,15.482143,15.910714,0.977534,0.97153,0.944637,0.919192,false,false,false,false,-0.510841,0.996681,-0.507273,0.996727,-0.49116,0.996939,-0.476039,0.997133
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-03-09,-9.52104,0.87599,23.608457,23.5673,22.517793,21.900436,21.283079,1.001746,1.046608,1.076111,1.107326,true,true,true,true,-9.644807,0.678212,-10.878621,0.600589,-11.942942,0.520404,-13.238105,0.369179
2026-03-10,4.8444,-0.1607,25.137354,25.0953,22.908514,21.646907,20.3853,1.001676,1.095457,1.159302,1.231049,true,true,true,true,4.85286,0.711125,5.070187,0.63104,5.213112,0.547534,5.364329,0.388982
2026-03-11,-4.55936,-0.12552,23.5823,23.5823,22.381282,21.701461,21.021639,1.0,1.053662,1.086669,1.121811,false,true,true,true,-4.567,0.678648,-4.994584,0.599522,-5.285675,0.518593,-5.612795,0.36715


In [ ]:
short = short.with_columns((~pl.col("30/60 flag")).alias("inverse 30/60 flag"))
short = add_cumret(short, "inverse 30/60 flag", "BATS:VIXY - Return")

In [ ]:
# check to see if the greater the ratio of front month : back month, the higher the return on VIXY 